# 🚨 SRE Incident-Response Agent: SFT + GRPO Training

**End-to-end LLM + RL pipeline** for training an autonomous incident-response agent.

| Component | Detail |
|-----------|--------|
| **Model** | `Qwen/Qwen2.5-3B` (4-bit quantized via BitsAndBytes) |
| **Phase 1** | Supervised Fine-Tuning (SFT) on expert trajectories |
| **Phase 2** | GRPO-style RL against a live environment |
| **Environment** | [incident-response-env](https://github.com/Praneeth0910/incident-response-env) — 16 SRE tasks |
| **Hardware** | Kaggle T4 GPU (16 GB VRAM) |

---

### Notebook Structure
1. 📦 Environment & Dependency Setup
2. 🔍 Environment Exploration
3. 📊 Synthetic SFT Dataset Generation
4. 🧠 Model Loading (Qwen2.5-3B, 4-bit)
5. 🎯 Phase 1 — SFT Training
6. 🎮 Phase 2 — GRPO-Style RL Training
7. 📈 Training Curves & Analysis
8. 🧪 Evaluation on Unseen Tasks
9. 💾 Model Export

---
## 1 · 📦 Environment & Dependency Setup

In [ ]:
%%capture
# ── Clone the incident-response-env repository ──
!rm -rf incident-response-env
!git clone https://github.com/Praneeth0910/incident-response-env.git

# ── Install dependencies ──
!pip install -q \
    pydantic>=2.7 \
    transformers>=4.45 \
    accelerate>=0.30 \
    bitsandbytes>=0.43 \
    peft>=0.12 \
    trl>=0.12 \
    datasets \
    matplotlib \
    httpx \
    openai

print("Dependencies installed")

In [ ]:
import sys, os, json, random, warnings, time, gc
import numpy as np
warnings.filterwarnings("ignore")

# Add the repo to Python path
REPO_DIR = "incident-response-env"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    _props = torch.cuda.get_device_properties(0)
    _total = getattr(_props, 'total_memory', 0) / 1e9
    print(f"VRAM:     {_total:.1f} GB")
else:
    print("WARNING: No GPU detected!")

---
## 2 · 🔍 Environment Exploration

Understand the action space, observation structure, and task definitions.

In [ ]:
from environment import IncidentResponseEnv, TASKS, SERVICES
from models import Action, Observation, Reward

print("=== Incident Response Environment ===")
print(f"Services : {SERVICES}")
print(f"Tasks    : {len(TASKS)}")
print()

# ── Action space ──
ACTION_TYPES = [
    "read_logs", "check_metrics", "check_health",
    "run_db_query", "restart_service", "rollback_deployment", "declare_rca"
]
print(f"Action types ({len(ACTION_TYPES)}): {ACTION_TYPES}")
print(f"Targets      ({len(SERVICES)}):      {SERVICES}")
print()

# ── Show all tasks ──
for tid, t in TASKS.items():
    print(f"  {tid:35s} | {t['difficulty']:6s} | steps={t['max_steps']:2d} | fault={t['fault_service']}:{t['fault_type']}")

In [ ]:
# ── Run a sample episode with manual actions to understand the API ──
env = IncidentResponseEnv()
obs = env.reset("task_cpu_spike", seed=42)
print(f"Alert:   {obs.alert}")
print(f"Message: {obs.message}")
print()

# Optimal path: observe → measure → confirm → fix → declare
demo_actions = [
    Action(action_type="read_logs",       target="auth-service"),
    Action(action_type="check_metrics",   target="auth-service"),
    Action(action_type="check_health",    target="auth-service"),
    Action(action_type="restart_service", target="auth-service"),
    Action(action_type="declare_rca",     target="auth-service"),
]

total_reward = 0
for a in demo_actions:
    obs, rew, done, info = env.step(a)
    total_reward += rew.value
    print(f"  Step {info['step']}: {a.action_type:20s} -> {a.target:20s} | r={rew.value:+.3f} | cum={info['cumulative_reward']:+.3f}")
    if done:
        break

print(f"\nTotal reward: {total_reward:.3f}  |  RCA correct: {env._rca_correct}")

---
## 3 · 📊 Synthetic SFT Dataset Generation

Generate expert-quality incident-response trajectories using a rule-based expert agent.
Each trajectory becomes a multi-turn ChatML conversation for SFT.

In [ ]:
sys.path.insert(0, "training")
from expert_agent import ExpertAgent

# ── Generate expert trajectories for each task ──
EPISODES_PER_TASK = 3
trajectories = []

# Skip multi-root-cause task (expert can't reliably solve it)
TRAIN_TASKS = [t for t in TASKS if t not in ("task_expert", "task_expert_long_horizon")]

print(f"Generating {EPISODES_PER_TASK} episodes x {len(TRAIN_TASKS)} tasks...\n")
for task_id in TRAIN_TASKS:
    task = TASKS[task_id]
    for ep in range(EPISODES_PER_TASK):
        try:
            expert = ExpertAgent(task)
            traj = expert.run_episode(IncidentResponseEnv(), task_id, seed=ep)
            trajectories.append(traj)
        except Exception as e:
            print(f"  [WARN] {task_id} ep {ep}: {e}")

rca_ok = sum(1 for t in trajectories if t.rca_correct)
print(f"\nGenerated {len(trajectories)} trajectories")
print(f"RCA correct: {rca_ok}/{len(trajectories)} ({100*rca_ok/len(trajectories):.0f}%)")
print(f"Tasks covered: {len(set(t.task_id for t in trajectories))}")

In [ ]:
# ── Format trajectories into ChatML for SFT ──

SYSTEM_PROMPT = (
    "You are an expert SRE agent investigating a production incident. "
    "Analyze the observation and choose the best next action. "
    "Respond ONLY with valid JSON: {\"action_type\": \"...\", \"target\": \"...\", \"reasoning\": \"...\"}. "
    "action_type must be one of: [check_metrics, check_health, read_logs, "
    "restart_service, run_db_query, rollback_deployment, declare_rca]. "
    "target must be a service name from: [api-gateway, auth-service, order-service, "
    "notification-service, redis-cache, postgres-db]."
)

sft_samples = []

for traj in trajectories:
    if not traj.rca_correct:
        continue  # only learn from successful episodes

    for step in traj.steps:
        obs = step.get("observation", "")[:200]
        action_str = step.get("action", "check_health:unknown")

        # Parse "action_type:target"
        parts = action_str.split(":", 1)
        action_type = parts[0]
        target = parts[1] if len(parts) > 1 else "unknown"

        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n"
            f"[Task: {traj.task_id} | Step: {step['step']}]\n"
            f"{obs}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        reasoning = f"Based on the observation, {action_type} on {target} is the correct next action."
        response_json = json.dumps({
            "action_type": action_type,
            "target": target,
            "reasoning": reasoning
        })
        response = f"{response_json}<|im_end|>"

        sft_samples.append({"prompt": prompt, "response": response, "text": prompt + response})

print(f"SFT samples: {len(sft_samples)}")
print(f"\n--- Sample (first 500 chars) ---")
print(sft_samples[0]["text"][:500])

In [ ]:
# ── Dataset Quality Visualization ──
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 110})
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle("SFT Dataset Overview", fontsize=15, fontweight="bold")

# 3a. Reward distribution
rewards = [t.total_reward for t in trajectories if t.rca_correct]
axes[0].hist(rewards, bins=15, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvline(np.mean(rewards), color="red", ls="--", label=f"mean={np.mean(rewards):.2f}")
axes[0].set_xlabel("Total Reward")
axes[0].set_ylabel("Count")
axes[0].set_title("Episode Reward Distribution")
axes[0].legend()

# 3b. Episodes per task
task_counts = Counter(t.task_id.replace("task_", "") for t in trajectories if t.rca_correct)
names = sorted(task_counts.keys())
axes[1].barh(names, [task_counts[n] for n in names], color="#8172B2", edgecolor="white")
axes[1].set_xlabel("Episodes")
axes[1].set_title("Episodes per Task")

# 3c. Action type distribution
action_counts = Counter()
for s in sft_samples:
    parsed = json.loads(s["response"].replace("<|im_end|>", ""))
    action_counts[parsed["action_type"]] += 1
ax_names = sorted(action_counts.keys())
axes[2].bar(ax_names, [action_counts[a] for a in ax_names], color="#55A868", edgecolor="white")
axes[2].set_xlabel("Action Type")
axes[2].set_ylabel("Count")
axes[2].set_title("Action Distribution")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("dataset_overview.png", bbox_inches="tight")
plt.show()
print("Saved -> dataset_overview.png")

---
## 4 · 🧠 Model Loading (Qwen2.5-3B, 4-bit)

Load the base model with NF4 quantization to fit comfortably on T4 GPU (16 GB).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-3B"
MAX_SEQ_LEN = 512

# ── 4-bit quantization config ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} with 4-bit quantization...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# ── Tokenizer setup ──
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size : {len(tokenizer)}")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"VRAM used  : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# ── Add LoRA adapter ──
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

---
## 5 · 🎯 Phase 1 — SFT Training

Fine-tune the model on expert trajectories using TRL's SFTTrainer.

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer

# ── Build HF dataset ──
sft_dataset = Dataset.from_list(sft_samples)
sft_split = sft_dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(sft_split['train'])} | Eval: {len(sft_split['test'])}")

# ── Metrics callback for plotting ──
class MetricsLogger(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_loss = []
        self.lr_history = []
        self.start_time = None

    def on_train_begin(self, args, state, control, **kw):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kw):
        if logs is None:
            return
        s = state.global_step
        if "loss" in logs:
            self.train_loss.append((s, logs["loss"]))
        if "eval_loss" in logs:
            self.eval_loss.append((s, logs["eval_loss"]))
        if "learning_rate" in logs:
            self.lr_history.append((s, logs["learning_rate"]))

sft_logger = MetricsLogger()

# ── Training arguments (T4-optimized) ──
sft_args = TrainingArguments(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    max_grad_norm=1.0,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_split["train"],
    eval_dataset=sft_split["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    args=sft_args,
    callbacks=[sft_logger],
)

eff_batch = sft_args.per_device_train_batch_size * sft_args.gradient_accumulation_steps
print(f"Effective batch size: {eff_batch}")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
print("SFT Training started...")
sft_result = trainer.train()

elapsed = time.time() - sft_logger.start_time
print(f"\nSFT complete in {elapsed/60:.1f} min")
print(f"Final train loss: {sft_result.training_loss:.4f}")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# ── SFT Loss & LR Curves ──
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
fig.suptitle("Phase 1 - SFT Training Diagnostics", fontsize=14, fontweight="bold")

# Train loss
if sft_logger.train_loss:
    steps, losses = zip(*sft_logger.train_loss)
    axes[0].plot(steps, losses, alpha=0.4, color="#4C72B0", lw=0.8, label="raw")
    w = max(1, len(losses) // 8)
    smoothed = [np.mean(losses[max(0, i - w):i + 1]) for i in range(len(losses))]
    axes[0].plot(steps, smoothed, color="#C44E52", lw=2, label=f"smooth(w={w})")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Train Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No data", ha="center", va="center", transform=axes[0].transAxes)

# Eval loss
if sft_logger.eval_loss:
    es, el = zip(*sft_logger.eval_loss)
    axes[1].plot(es, el, "o-", color="#55A868", lw=2, ms=8)
    for s, l in zip(es, el):
        axes[1].annotate(f"{l:.3f}", (s, l), textcoords="offset points", xytext=(0, 12), ha="center")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Eval Loss")
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "No data", ha="center", va="center", transform=axes[1].transAxes)

# Learning rate
if sft_logger.lr_history:
    ls, lv = zip(*sft_logger.lr_history)
    axes[2].plot(ls, lv, color="#8172B2", lw=2)
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("LR")
    axes[2].set_title("Learning Rate Schedule")
    axes[2].ticklabel_format(axis="y", style="sci", scilimits=(-4, -4))
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "No data", ha="center", va="center", transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig("sft_curves.png", bbox_inches="tight")
plt.show()
print("Saved -> sft_curves.png")

---
## 6 · 🎮 Phase 2 — GRPO-Style RL Training

**Group Relative Policy Optimization (GRPO)** loop:
- For each state, sample `G` candidate actions from the model
- Execute each in the environment to get rewards
- Compute advantage = (reward - mean) / std across the group
- Update policy to upweight high-advantage actions

In [ ]:
# ── Helper functions for RL ──

def state_to_prompt(task_id, alert, obs_message, step_num, max_steps):
    """Convert environment state into a ChatML prompt for the model."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"[Task: {task_id} | Step: {step_num}/{max_steps}]\n"
        f"Alert: {alert[:150]}\n"
        f"Observation: {obs_message[:200]}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def parse_model_action(text):
    """Parse model output JSON into an Action object. Falls back to safe defaults."""
    try:
        clean = text.split("<|im_end|>")[0].strip()
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start >= 0 and end > start:
            parsed = json.loads(clean[start:end])
            action_type = parsed.get("action_type", "check_health")
            target = parsed.get("target", "api-gateway")
            if action_type not in ACTION_TYPES:
                action_type = "check_health"
            if target not in SERVICES:
                target = SERVICES[0]
            return Action(action_type=action_type, target=target)
    except Exception:
        pass
    return Action(action_type=random.choice(ACTION_TYPES[:3]), target=random.choice(SERVICES))


@torch.no_grad()
def generate_action(model, tokenizer, prompt, temperature=0.7):
    """Generate a single action from the model."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=max(temperature, 0.01),
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return text, new_tokens


print("RL helpers defined")

In [ ]:
# ── GRPO RL config ──
from torch.optim import AdamW
from torch.cuda.amp import autocast

RL_EPISODES = 30
GROUP_SIZE = 3
RL_LR = 1e-5
MAX_STEPS_PER_EP = 6
CLIP_GRAD = 1.0

RL_TASKS = [
    "task_cpu_spike", "task_disk_full", "task_memory_leak",
    "task_db_connection_leak", "task_ssl_cert_expired", "task_k8s_pod_crashloop"
]

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=RL_LR,
    weight_decay=0.01,
)

# Tracking arrays
rl_rewards = []
rl_losses = []
rl_rca_correct = []
rl_parse_success = []

print(f"GRPO RL Config:")
print(f"  Episodes   : {RL_EPISODES}")
print(f"  Group size : {GROUP_SIZE}")
print(f"  RL tasks   : {len(RL_TASKS)}")
print(f"  LR         : {RL_LR}")
print(f"  Max steps  : {MAX_STEPS_PER_EP}")

In [ ]:
print("GRPO RL Training started...\n")
rl_start = time.time()

for episode in range(RL_EPISODES):
    task_id = RL_TASKS[episode % len(RL_TASKS)]
    task = TASKS[task_id]
    env = IncidentResponseEnv()
    obs = env.reset(task_id, seed=episode)

    episode_reward = 0.0
    episode_loss = 0.0
    episode_steps = 0
    parse_ok = 0
    total_actions = 0
    done = False

    for step_idx in range(MAX_STEPS_PER_EP):
        if done:
            break

        prompt = state_to_prompt(
            task_id, task["alert"], obs.message, step_idx + 1, MAX_STEPS_PER_EP
        )

        # ── GRPO: Generate G candidate actions ──
        group_actions = []
        group_rewards = []
        group_texts = []

        for g in range(GROUP_SIZE):
            action_text, _ = generate_action(model, tokenizer, prompt, temperature=0.7 + 0.1 * g)
            action = parse_model_action(action_text)
            total_actions += 1

            # Evaluate in a fresh env copy
            test_env = IncidentResponseEnv()
            test_env.reset(task_id, seed=episode)
            try:
                _, test_rew, _, _ = test_env.step(action)
                r = test_rew.value
            except Exception:
                r = -0.5

            group_actions.append(action)
            group_rewards.append(r)
            group_texts.append(action_text)

            # Track parse success
            try:
                clean = action_text.split("<|im_end|>")[0].strip()
                s_idx, e_idx = clean.find("{"), clean.rfind("}") + 1
                if s_idx >= 0 and e_idx > s_idx:
                    json.loads(clean[s_idx:e_idx])
                    parse_ok += 1
            except Exception:
                pass

        # ── Compute GRPO advantage ──
        group_rewards_t = torch.tensor(group_rewards, dtype=torch.float32)
        mean_r = group_rewards_t.mean()
        std_r = group_rewards_t.std() + 1e-8
        advantages = (group_rewards_t - mean_r) / std_r

        # ── Select best action, apply reward-weighted policy gradient ──
        best_idx = advantages.argmax().item()
        best_action = group_actions[best_idx]
        best_text = group_texts[best_idx]
        best_advantage = advantages[best_idx].item()

        if abs(best_advantage) > 0.1:
            model.train()
            full_text = prompt + best_text
            enc = tokenizer(
                full_text, return_tensors="pt", truncation=True,
                max_length=MAX_SEQ_LEN, padding=False
            ).to(model.device)

            with autocast(dtype=torch.bfloat16):
                out = model(**enc, labels=enc["input_ids"])
                loss = out.loss * (-best_advantage)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], CLIP_GRAD
            )
            optimizer.step()
            optimizer.zero_grad()
            episode_loss += loss.item()
            model.eval()

        # ── Execute best action in real episode ──
        try:
            obs, rew, done, info = env.step(best_action)
            episode_reward += rew.value
        except Exception as e:
            break

        episode_steps += 1

    # ── Log episode ──
    rl_rewards.append(episode_reward)
    rl_losses.append(episode_loss / max(episode_steps, 1))
    rl_rca_correct.append(env._rca_correct)
    rl_parse_success.append(parse_ok / max(total_actions, 1))

    if (episode + 1) % 5 == 0 or episode == 0:
        win = min(5, episode + 1)
        avg_r = np.mean(rl_rewards[-win:])
        avg_rca = np.mean(rl_rca_correct[-win:])
        avg_parse = np.mean(rl_parse_success[-win:])
        print(
            f"  Ep {episode+1:3d}/{RL_EPISODES} | {task_id:30s} | "
            f"r={episode_reward:+.3f} | avg5={avg_r:+.3f} | "
            f"RCA={avg_rca:.0%} | parse={avg_parse:.0%} | "
            f"loss={rl_losses[-1]:.4f}"
        )

    # Free memory every 10 episodes
    if (episode + 1) % 10 == 0:
        gc.collect()
        torch.cuda.empty_cache()

rl_elapsed = time.time() - rl_start
print(f"\nGRPO RL complete in {rl_elapsed/60:.1f} min")
print(f"Avg reward (last 10): {np.mean(rl_rewards[-10:]):+.3f}")
print(f"RCA success (last 10): {np.mean(rl_rca_correct[-10:]):.0%}")

---
## 7 · 📈 Training Curves & Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Phase 2 - GRPO RL Training Results", fontsize=15, fontweight="bold")

# 7a. Episode Rewards
ax = axes[0, 0]
ax.plot(rl_rewards, alpha=0.4, color="#4C72B0", lw=1, label="per-episode")
w = max(1, len(rl_rewards) // 6)
if len(rl_rewards) > w:
    smoothed = [np.mean(rl_rewards[max(0, i - w):i + 1]) for i in range(len(rl_rewards))]
    ax.plot(smoothed, color="#C44E52", lw=2.5, label=f"rolling avg (w={w})")
ax.axhline(0, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.set_title("Episode Reward")
ax.legend()
ax.grid(alpha=0.3)

# 7b. Policy Loss
ax = axes[0, 1]
ax.plot(rl_losses, color="#DD8452", lw=1.5)
ax.set_xlabel("Episode")
ax.set_ylabel("Avg Loss")
ax.set_title("RL Policy Loss")
ax.grid(alpha=0.3)

# 7c. RCA Success Rate (rolling)
ax = axes[1, 0]
if len(rl_rca_correct) > 3:
    rolling = [np.mean(rl_rca_correct[max(0, i - 4):i + 1]) for i in range(len(rl_rca_correct))]
    ax.plot(rolling, color="#55A868", lw=2.5)
    ax.fill_between(range(len(rolling)), rolling, alpha=0.2, color="#55A868")
ax.set_xlabel("Episode")
ax.set_ylabel("Success Rate")
ax.set_title("RCA Success Rate (rolling 5)")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# 7d. JSON Parse Success Rate
ax = axes[1, 1]
ax.plot(rl_parse_success, color="#937860", lw=1.5)
ax.set_xlabel("Episode")
ax.set_ylabel("Parse Success")
ax.set_title("JSON Parse Rate")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("rl_training_curves.png", bbox_inches="tight")
plt.show()
print("Saved -> rl_training_curves.png")

---
## 8 · 🧪 Evaluation on Unseen Tasks

Test the trained model on tasks it has never seen during RL training.

In [ ]:
EVAL_TASKS = [
    t for t in TASKS
    if t not in RL_TASKS and t not in ("task_expert", "task_expert_long_horizon")
]
print(f"Evaluating on {len(EVAL_TASKS)} unseen tasks:\n")

model.eval()
eval_results = []

for task_id in EVAL_TASKS:
    task = TASKS[task_id]
    env = IncidentResponseEnv()
    obs = env.reset(task_id, seed=999)

    ep_reward = 0.0
    actions_taken = []
    done = False

    for step_idx in range(min(task["max_steps"], 8)):
        if done:
            break

        prompt = state_to_prompt(
            task_id, task["alert"], obs.message, step_idx + 1, 8
        )

        action_text, _ = generate_action(model, tokenizer, prompt, temperature=0.3)
        action = parse_model_action(action_text)

        try:
            obs, rew, done, info = env.step(action)
            ep_reward += rew.value
            actions_taken.append(f"{action.action_type}:{action.target}")
        except Exception as e:
            actions_taken.append(f"ERROR: {e}")
            break

    result = {
        "task_id": task_id,
        "fault": f"{task['fault_service']}:{task['fault_type']}",
        "reward": ep_reward,
        "rca_correct": env._rca_correct,
        "actions": actions_taken,
    }
    eval_results.append(result)

    icon = "Pass" if env._rca_correct else "Fail"
    print(f"[{icon}] {task_id:35s} | reward={ep_reward:+.3f} | steps={len(actions_taken)}")
    for a in actions_taken:
        print(f"       -> {a}")
    print()

# Summary
total_eval = len(eval_results)
correct_eval = sum(1 for r in eval_results if r["rca_correct"])
avg_reward_eval = np.mean([r["reward"] for r in eval_results])
print("=" * 60)
print(f"Evaluation Summary:")
print(f"  Tasks evaluated : {total_eval}")
print(f"  RCA correct     : {correct_eval}/{total_eval} ({100 * correct_eval / max(total_eval, 1):.0f}%)")
print(f"  Avg reward      : {avg_reward_eval:+.3f}")
print("=" * 60)

---
## 9 · 💾 Model Export

In [ ]:
SAVE_DIR = "./sre_agent_final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Print saved files
saved_files = []
total_bytes = 0
for f in sorted(os.listdir(SAVE_DIR)):
    fp = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fp):
        sz = os.path.getsize(fp)
        saved_files.append((f, sz))
        total_bytes += sz

total_mb = total_bytes / 1e6
print(f"Model saved to {SAVE_DIR}")
print(f"Files: {len(saved_files)} | Size: {total_mb:.1f} MB")
for fname, sz in saved_files:
    print(f"  {fname:40s} {sz / 1e6:>6.2f} MB")

In [ ]:
# ── Final Summary ──
print()
print("=" * 70)
print("  SRE INCIDENT-RESPONSE AGENT - TRAINING COMPLETE")
print("=" * 70)

summary_lines = [
    f"  Model:         {MODEL_NAME}",
    f"  Method:        SFT + GRPO-style RL",
    f"  SFT samples:   {len(sft_samples)}",
    f"  SFT loss:      {sft_result.training_loss:.4f}",
    f"  RL episodes:   {RL_EPISODES}",
    f"  RL avg reward: {np.mean(rl_rewards[-10:]):+.3f} (last 10)",
    f"  RL RCA rate:   {np.mean(rl_rca_correct[-10:]):.0%} (last 10)",
    f"  Eval RCA:      {correct_eval}/{total_eval} ({100 * correct_eval / max(total_eval, 1):.0f}%)",
    f"  Eval reward:   {avg_reward_eval:+.3f}",
    f"  Model saved:   {SAVE_DIR} ({total_mb:.1f} MB)",
    f"  VRAM used:     {torch.cuda.memory_allocated(0) / 1e9:.2f} GB",
]
for line in summary_lines:
    print(line)

print("=" * 70)
print("\nDone!")